# 환경설정

In [1]:
# 라이브러리 설치
# pip install langchain-openai langchain-core langgraph langchain-chroma rank_bm25

import re
import os
import logging
import json
import pymongo
from pymongo import MongoClient
from datetime import datetime
from dotenv import load_dotenv
import pandas as pd
from typing import Literal, TypedDict, List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain.retrievers import EnsembleRetriever
from langgraph.graph import StateGraph, START, END
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers import ContextualCompressionRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [3]:
# 환경 변수 로드 및 로깅 설정
load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 로그 저장 디렉토리 설정
LOG_DIR = "chat_logs"
os.makedirs(LOG_DIR, exist_ok=True)

"""# MongoDB 클라이언트 설정
MONGO_IP = os.getenv("MONGO_IP")
MONGO_PORT = int(os.getenv("MONGO_PORT"))
MONGO_USER = os.getenv("MONGO_USER")
MONGO_PASSWORD = os.getenv("MONGO_PASSWORD")

# 연결 URI 생성
mongo_uri = f"mongodb://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_IP}:{MONGO_PORT}/?authSource=admin"
client = MongoClient(mongo_uri)

# 사용할 데이터베이스와 컬렉션 지정
db = client['chatbot_db']
collection = db['chat_logs']
"""

'# MongoDB 클라이언트 설정\nMONGO_IP = os.getenv("MONGO_IP")\nMONGO_PORT = int(os.getenv("MONGO_PORT"))\nMONGO_USER = os.getenv("MONGO_USER")\nMONGO_PASSWORD = os.getenv("MONGO_PASSWORD")\n\n# 연결 URI 생성\nmongo_uri = f"mongodb://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_IP}:{MONGO_PORT}/?authSource=admin"\nclient = MongoClient(mongo_uri)\n\n# 사용할 데이터베이스와 컬렉션 지정\ndb = client[\'chatbot_db\']\ncollection = db[\'chat_logs\']\n'

In [4]:
#----1. 모델 정의----
model = ChatOpenAI(
    model_name='gpt-4o-mini',
    temperature=0
)

# 1. 전처리1

### 1-1. txt -> xlsx

In [ ]:
import os
import pandas as pd

def load_txt_to_df(folder_path):
    """폴더 안의 txt 파일들을 읽어 DataFrame 생성"""
    data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            data.append([file_name, content])
    return pd.DataFrame(data, columns=["file_name", "content"])

# 폴더 경로 설정
folder_path_1 = "folder_path_1"
folder_path_2 = "folder_path_2"

# 각각 DataFrame 생성
df1 = load_txt_to_df(folder_path_1)
df2 = load_txt_to_df(folder_path_2)

# Excel 파일로 저장
df1.to_excel("df1.xlsx", index=False, engine="openpyxl")
df2.to_excel("df2.xlsx", index=False, engine="openpyxl")

### 1-2. 시스템 메시지 + 이모티콘 제거

In [ ]:
# (필요시) 데이터 불러오기
df1 = pd.read_excel("df1.xlsx")
df2 = pd.read_excel("df2.xlsx")

def clean_chat_text(text: str) -> str:
    # 1. 불필요한 안내 문구 제거
    text = re.sub(r"Messages and calls are end-to-end encrypted.*?\n", "", text)
    text = re.sub(r"Welcome to the chat:.*?\n", "", text)

    # 2. 특수문자/이모지 제거
    text = re.sub(r"[^\w\s\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]", " ", text)  
    # (아랍/파슈토/다리어 문자 범위는 보존)

    return text

# content 열 전처리 적용
df1["clean_content"] = df1["content"].apply(clean_chat_text)
df2["clean_content"] = df2["content"].apply(clean_chat_text)

# 중복 제거
df1 = df1.drop_duplicates(subset=["clean_content"]).reset_index(drop=True)
df2 = df2.drop_duplicates(subset=["clean_content"]).reset_index(drop=True)

# 저장
#df1.to_excel("df1_cleaned.xlsx", index=False, engine="openpyxl")

### 1-3. 기계적인 메시지 삭제

### 1-4. AI 활용하여 사적인 대화 판별 후 삭제

In [10]:
df1 = pd.read_excel("df1_cleaned-2.xlsx")
data = df1

system = """
당신은 데이터 감별사입니다. 입력받은 파일의 데이터 중 사적인 데이터를 감별하여 반환합니다. 
사적인 대화는 모르는 사람과 대화하는 것이 아닌, 친근한 대화를 의미합니다.
사적인 대화라고 판단된다면 해당하는 행의 번호를 반환합니다. 
"""

prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{data}")])
chain = prompt | model | StrOutputParser()

out = chain.invoke({"data": data})
out

2025-09-09 22:25:21,956 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'사적인 대화로 판단되는 행의 번호는 다음과 같습니다:\n\n0, 1, 2, 3, 4, 512, 513, 514, 515, 516'